## How to train your first FiMMIA model on a new dataset


#### Installation
Download repository and install requirements.



In [ ]:
! git clone https://github.com/ai-forever/data_leakage_detect
! cd data_leakage_detect
! pip install -r requirements.txt


### Example
For example of usage of FiMMIA we take dataset [MERA-evaluation/CommonVideoQA](https://huggingface.co/datasets/MERA-evaluation/CommonVideoQA), train FiMMIA neural network model and will see inference.

#### Train FiMMIA model on dataset MERA-evaluation/CommonVideoQA
We divided the training algorithm into the following subsequent steps with some modifications:
* [Data preparation](#Data-preparation)
* [SFT MLLM finetuning](#SFT-MLLM-finetuning)
* [Neighbor generation](#Neighbor-generation)
* [Embedding generation](#Embedding-generation)
* [Loss computation](#Loss-computation)
* [Training the attack model](#Training-the-attack-model)
* [Run MIA inference](#Run-MIA-inference)

#### Data preparation

Download data and convert to own working format. We should convert our dataset into pandas format with following structure:

| input | answer | video | ds_name  |
|----------|--------|-------|----------|

* `input` example:

```text
Очень бы хотелось получить решение такой задачи. В этой задаче необходимо выбрать правильный ответ из четырех предложенных вариантов на основе переданных вопроса и видео.

Имеется 1 видеофайл

Желательно, чтобы вы ознакомились с данными и решили задачу, выбрав из вариантов ответа один или несколько правильных.

Видеофайл: <video>
Вопрос:
Почему баскетболист отдал мяч игроку в майке другого цвета?

A. Он перепутал соперника с игроком своей команды.
B. Эти игроки тренируются вместе, цвет одежды не имеет значения.
C. Он решил так поступить, потому что его команда всё равно проигрывает.
D. Он специально подыгрывает другой команде.

Первому из предложенных вариантов ответа присваивается литера А, второму литера B, третьему литера C и так далее по английскому алфавиту. В качестве ответа будет правильно вывести литеру, соответствующую верному варианту ответа из предложенных. Это лучше сделать в таком формате: по завершении рассуждений пишется слово ОТВЕТ, затем через пробел выводится выбранная литера. ОТВЕТ: 
```

* `answer` example: 'B'.
* `video` - is the modality column. For audio we should put `audio`, for image - `image`.
* `ds_name` is the dataset name. For example `CommonVideoQA`.

In [11]:
from datasets import load_dataset
import numpy as np
from pathlib import Path
import warnings
import torch
import sys
import pathlib
import torchvision
from fractions import Fraction
import pandas as pd
sys.path.append("../")
warnings.filterwarnings("ignore")
def save_video(line, video_path, video_codec="libx264"):
    video_codec = "libx264"
    container = line["inputs"]["video"]
    video_fps = Fraction(line["inputs"]["video"].get_avg_fps())
    total_frames = len(container)
    container.seek(0)
    frames = list(container)
    video_array = torch.from_numpy(np.stack([x.asnumpy() for x in frames]))
    torchvision.io.write_video(
        filename=str(video_path),
        video_array=video_array,
        fps=video_fps,
        video_codec=video_codec
    )

In [12]:
working_dir = Path("../data/")
ds_name = "CommonVideoQA"
ds_dir = working_dir / ds_name
ds_samples_dir = ds_dir / "samples"
ds_samples_dir.mkdir(parents=True, exist_ok=True)

In [3]:
ds = load_dataset("MERA-evaluation/CommonVideoQA")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/MERA-evaluation/CommonVideoQA/resolve/main/README.md (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 32acd1e3-e05f-4669-92b9-809b4fbd7c41)')' thrown while requesting HEAD https://huggingface.co/datasets/MERA-evaluation/CommonVideoQA/resolve/main/README.md
Retrying in 1s [Retry 1/5].
Generating test split: 100%|██████████| 1200/1200 [01:03<00:00, 19.00 examples/s]


In [5]:
lines = []
idx = 0
count = 100
for line in ds["test"]:
    idx = line["meta"]["id"]
    video_path = ds_samples_dir / f"video_{idx}.mp4"
    if not video_path.exists():
        save_video(line, video_path)
    lines.append({
        "input": line["instruction"].format(**line["inputs"]) + "ОТВЕТ: ",
        "answer": line["outputs"],
        "video": video_path.resolve().as_posix(),
        "ds_name": ds_name
    })
    idx += 1
    if idx == count:
        break

In [13]:
df_path = working_dir / "train_all.csv"
df = pd.DataFrame(lines)

In [7]:
df.to_csv(str(df_path), index=False)

In [8]:
df = pd.read_csv(str(df_path))

In [9]:
df.head()

,input,answer,video,ds_name
0,Очень бы хотелось получить решение такой задач...,B,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA
1,В датасете к задаче идёт такой промпт:\n\nНеоб...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA
2,Привет! Поможешь?\n\nМне попалась такая задача...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA
3,Сформулирована задача.\n\nВ задаче требуется с...,D,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA
4,Видеофайл: <video>\nВопрос:\nКто или что движе...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA


#### Neighbor generation
For each original data point $(t, s)$ we generate $K=24$ perturbed "neighbors" $(t^k_\prime, s^k_\prime)$. We apply four different perturbation techniques:
* Random masking and predicting the masks with [ai-forever/FRED-T5-1.7B](https://huggingface.co/ai-forever/FRED-T5-1.7B) model
* Deletion of random tokens
* Duplication
* Swapping of random tokens

to the text $t$ with each technique applied several times. For **multimodal** data, set `--modality_column=video` (or `image`/`audio`) to also generate adversarial modality perturbations (e.g. frame drops for video); otherwise only text neighbors are produced.

**Run command:**

```bash
fimmia neighbors \
  --model_path="ai-forever/FRED-T5-1.7B" \
  --dataset_path="path/to/train.csv" \
  --max_text_len=4000
```

Here
* `model_path` - T5 model for masking-based text neighbor generation
* `dataset_path` - path to dataset CSV (must contain `input`, `answer`, and optionally a modality column)
* `max_text_len` - max text length in characters
* `modality_column` - optional; use `video`, `image`, or `audio` to generate modality perturbations (one per text neighbor); omit for text-only

After running, the CSV is updated with a **neighbors** column and, when `modality_column` is set, a **modality_neighbors** column. 

In [14]:
df = pd.read_csv(str(df_path))

In [15]:
df.head()

,input,answer,video,ds_name,neighbors
0,Очень бы хотелось получить решение такой задач...,B,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,['Очень бы хотелось получить решение такой зад...
1,В датасете к задаче идёт такой промпт:\n\nНеоб...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,['В датасете к задаче идёт такой промпт: Необх...
2,Привет! Поможешь?\n\nМне попалась такая задача...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,['Привет! Поможешь? _Здравствуйте! попалась та...
3,Сформулирована задача.\n\nВ задаче требуется с...,D,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,['Задача по математике задача. В задаче требуе...
4,Видеофайл: <video>\nВопрос:\nКто или что движе...,A,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,['Видеофайл: <video> Вопрос: Кто или что плав...


* **input** for sample is

Видеофайл: \<video\>

Вопрос:

Что происходит в видео с 00.510 по 11.460 секунды?

A. Помешивают ложкой <span style="color:blue">**разогревшееся блюдо**</span> в сковороде.

B. Помешивают <span style="color:blue">**ложкой**</span> готовое блюдо в кастрюле.

C. Разделяют <span style="color:blue">**замёрзший продукт**</span> ложкой в сковороде, помогая размораживаться.

D. Закрывают сковороду крышкой.

* **Random masking** neighbor

Видеофайл: \<video\>

Вопрос:

Что происходит в видео с 00.510 по 11.460 секунды?

A. Помешивают ложкой <span style="color:red">**ройбуш ройбуш и готовое блюдо**</span> в сковороде.

B. Помешивают <span style="color:red">**ройбуш и** </span>готовое блюдо в кастрюле.

C. Разделяют <span style="color:red">**ройбуш и ройбуш**</span> ложкой в сковороде, помогая размораживаться.

D. Закрывают сковороду крышкой.

---

* **Deletion** neighbor

Видеофайл: \<video\>

Вопрос:

Что происходит в видео с 00.510 по 11.460 секунды?

A. Помешивают ложкой ~~**разогревшееся**~~ блюдо в сковороде.

B. Помешивают ~~**ложкой**~~ готовое блюдо в кастрюле.

C. Разделяют замёрзший продукт ложкой в сковороде, ~~**помогая**~~ размораживаться.

D. Закрывают сковороду крышкой.

---

* **Duplication** neighbor

Видеофайл: \<video\>

Вопрос:

**Что** Что **происходит** происходит в видео с 00.510 по 11.460 секунды?

A. Помешивают ложкой разогревшееся **блюдо** блюдо в сковороде.

B. Помешивают **ложкой** ложкой готовое блюдо в кастрюле.

C. Разделяют **замёрзший** замёрзший продукт ложкой в сковороде, помогая размораживаться.

D. Закрывают сковороду крышкой.

---

* **Swapping** neighbor

<span style="color:blue">**блюдо**</span> \<video\>

<span style="color:green">**размораживаться.**</span>

Что происходит в видео с <span style="color:red">**B.**</span> по 11.460 секунды?

A. Помешивают ложкой разогревшееся <span style="color:blue">**Видеофайл:**</span> в сковороде.

<span style="color:red">**00.510**</span> Помешивают ложкой готовое блюдо в кастрюле.

C. Разделяют замёрзший продукт ложкой в сковороде, помогая <span style="color:green">**Вопрос:**</span>

D. Закрывают сковороду крышкой.

---

**Split data into train and test**

In [16]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.1)
train_df.to_csv(str(working_dir / "train.csv"), index=False)
test_df.to_csv(str(working_dir / "test.csv"), index=False)

---

#### SFT MLLM finetuning
For obtaining positive labels that indicates leak we finetune MLLM. Here we obtain two MLLMs: original model $\mathcal{M}$ without leak and $\mathcal{M}_{leak}$ with leak.

**Running command** (video example; use `fimmia sft image` or `fimmia sft audio` for other modalities)

```bash
fimmia sft video \
  --train_df_path="path/to/train.csv" \
  --test_df_path="path/to/test.csv" \
  --num_train_epochs=5 \
  --model_id="Qwen/Qwen2.5-VL-3B-Instruct" \
  --output_dir="data/models/sft/Qwen2.5-VL-3B-Instruct"
```

#### Embedding generation
Then for each original text $t$ and its neighbors $t^k_\prime$ we extract their text embeddings (and optionally modality embeddings) in a single pass using the unified pipeline (`fimmia.embeds_joint`), which uses the **embedding_models** abstraction (BaseEmbedder): default text embedder is SentenceTransformer, default modality embedder is ImageBind. When `--modality_key` is set, both models are loaded **before** dataset processing.
$$e=\mathcal{E}(t), \quad e_{k}^{\prime} = \mathcal{E}(t_k^{\prime})$$
where $\mathcal{E}$ is [intfloat/e5-mistral-7b-instruct](https://huggingface.co/intfloat/e5-mistral-7b-instruct) for text. For multimodal data, set `--modality_key=video` (or `image`/`audio`) to compute modality embeddings in the same run.

**Run command:**

```bash
fimmia embeds \
  --df_path="data/train_all.csv" \
  --embed_model="intfloat/e5-mistral-7b-instruct" \
  --max_seq_length=4000 \
  --user_answer=0 \
  --device=cuda \
  --part_size=5000 \
  --run_single_file=1
```

Here
* `df_path` - path to dataset CSV (with `neighbors` column from the neighbor step)
* `embed_model` - text embedder model name (default: intfloat/e5-mistral-7b-instruct)
* `max_seq_length` - max sequence length for the text encoder (default: 4096)
* `user_answer` - 1 = neighbor text is input+neighbor, 0 = neighbor+answer (default: 0)
* `modality_key` - optional; use `video`, `image`, or `audio` to compute modality embeddings in the same pass; omit for text-only
* `device` - device for text and modality models (default: cuda)
* `part_size` - lines per output part file (default: 5000)
* `run_single_file` - 1 = process only df_path, 0 = process all CSVs under {df_path stem}_ng_parts/ (default: 1)

Output: `embeds/` is written under the same directory as the dataset (e.g. `data/train_all/embeds/`), with `part_0.csv`, `part_1.csv`, ... Each part has **neighbor_embeds** and **input_embeds** (text). When `--modality_key=video` is set, `video_embeds/` is also written with **neighbor_video_embeds** and **input_video_embeds**. **input_embeds** is non-NaN only on the first row of each original sample (e.g. id=0, id=25).

In [19]:
df = pd.read_csv("../data/test/embeds/part_0.csv")

In [20]:
df.head(2)

,input,answer,video,ds_name,neighbor,neighbor_embeds,input_embeds
0,Внимание!\n\nВ датасете к задаче идёт такой пр...,B,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,Внимание! В датасете к задаче идёт такой промп...,"[0.0013391461689025164, -0.013676884584128857,...","[0.002668527187779546, -0.014232771471142769, ..."
1,Внимание!\n\nВ датасете к задаче идёт такой пр...,B,/home/jovyan/emelyanov/data_leakage_detect/dat...,CommonVideoQA,Внимание! датасете к задаче такой промпт: Общи...,"[0.003614989807829261, -0.013492275960743427, ...",NaN


---

#### Loss computation

We compute the multimodal loss for both models $\mathcal{M}$ and $\mathcal{M}_{leak}$ on both the original and neighbor data points:
$$\mathcal{L} = \mathcal{L}(\mathcal{M}, t, s), \quad \mathcal{L}_k^{\prime} = \mathcal{L}(\mathcal{M}, t^k_{\prime}, s^k_{\prime})$$
Text and modality (image/video/audio) are passed to the MLLM. When the dataset has a **modality_neighbors** column (from neighbor generation with `--modality_column`), each neighbor row uses the corresponding perturbed modality path; otherwise the original modality is used for all.

**Run command:**

```bash

# For video modality
fimmia loss video \
  --model_id=Qwen/Qwen2.5-VL-3B-Instruct \
  --model_name=Qwen/Qwen2.5-VL-3B-Instruct \
  --label=0 \
  --df_path="path/to/train.csv"

# For audio modality
fimmia loss audio \
  --model_id=Qwen/Qwen2-Audio-7B-Instruct \
  --model_name=Qwen2-Audio-7B-Instruct \
  --label=0 \
  --df_path="path/to/train.csv"

# For image modality (--model_name is the model path for from_pretrained)
fimmia loss image \
  --model_name=Qwen/Qwen2.5-VL-3B-Instruct \
  --label=0 \
  --df_path="path/to/train.csv"
```
Here
* `model_id` - path to MLLM
* `model_name` - name of MLLM model (using for store results)
* `label` - label of dataset `0` for no lean or `1` for leak model (0 by default)
* `df_path` - path to dataset for calculating loss
* `part_size` - lines for split dataframe into smaller frames

After running this command we will create new directory with name `data/train_all/loss/Qwen2.5-VL-3B-Instruct/leak` or `data/train_all/loss/Qwen2.5-VL-3B-Instruct/no_leak` depends from `label`. Directory contains list of files `part_0.csv`, `part_1.csv` and so on depends from dataset size. Each file has two new columns:
* **neighbor_loss** - loss for neighbor: neighbor text + (perturbed or original) modality
* **input_loss** - loss for original sample: text + modality

For training we should produce files for both $\mathcal{M}$ and $\mathcal{M}_{leak}$ models.

For finetunned sft model **run command**:

```bash
fimmia loss video \
  --model_id=data/Qwen2.5-VL-3B-Instruct \
  --model_name=Qwen2.5-VL-3B-Instruct \
  --label=0 \
  --df_path="path/to/train.csv"
```

In [ ]:
df = pd.read_csv("data/train/loss/Qwen2.5-VL-3B-Instruct/no_leak/part_0.csv")

In [ ]:
df["video"] = [str(ds_samples_dir / Path(x).name) for x in df["video"]]

In [123]:
df.head(5)

,input,answer,video,ds_name,neighbor,neighbor_loss,input_loss
0,Видеофайл: <video>\nВопрос:\nЧто происходит в ...,A,data/CommonVideoQA/samples/video419.mp4,CommonVideoQA,Видеофайл: <video> Что Вопрос: происходит в ви...,0.872269,0.863068
1,Видеофайл: <video>\nВопрос:\nЧто происходит в ...,A,data/CommonVideoQA/samples/video419.mp4,CommonVideoQA,Видеофайл: <video> Вопрос: Что происходит в ви...,0.935770,0.863068
2,Видеофайл: <video>\nВопрос:\nЧто происходит в ...,A,data/CommonVideoQA/samples/video419.mp4,CommonVideoQA,Видеофайл: <video> Вопрос: Что в видео с 00.51...,0.862926,0.863068
3,Видеофайл: <video>\nВопрос:\nЧто происходит в ...,A,data/CommonVideoQA/samples/video419.mp4,CommonVideoQA,Видеофайл: <video> Вопрос: происходит видео с ...,1.260783,0.863068
4,Видеофайл: <video>\nВопрос:\nЧто происходит в ...,A,data/CommonVideoQA/samples/video419.mp4,CommonVideoQA,Видеофайл: <video> Вопрос: Что происходит сков...,1.256082,0.863068


---

#### Training the attack model
The core of FiMMIA is a binary neural network classifier trained to distinguish between models that have and have not seen the data. For each neighbor $k$ we create two training examples by computing feature differences:
$$\Delta \mathcal{L} = \mathcal{L} - \mathcal{L}^k_{\prime}, \quad \Delta e = e - e^{k}_{\prime}$$

These feature vectors are paired with labels $y \in \{0, 1\}$ indicating whether the losses came from $\mathcal{M}$ (non-leaked) or $\mathcal{M}_{leak}$ (leaked). However, absolute values of these statistics may vary across datasets and models. To make the system more stable, we apply the z-score normalization technique. During the training phase, we calculate the mean $\mu$ and standard deviation $\sigma$ for the models loss differences $\Delta \mathcal{L}$ using the evaluation data. 

$$ \Delta \mathcal{L}_{norm} = \frac{\Delta \mathcal{L}-\mu}{\sigma} $$

This process yields random batch training triplets $(\Delta \mathcal{L}_{norm}, \Delta e, y)$ per original data point. The FiMMIA detector, $f_{FiMMIA}$ is trained to predict the probability $p=f_{FiMMIA}(\Delta \mathcal{L}_{norm}, \Delta e)$ that the input features originate from a model that has been trained on the target data.

##### The detailed architecture of the FiMMIA is provided below.

1. **Input Data**
* *loss\_input* A tensor fed into the *loss\_component*.
* *embedding\_input* A tensor fed into the *embedding\_component*.

2. **loss\_component**:
* A Linear layer: 1 input feature $\rightarrow$ *projection\_size* output features.
* *Dropout(0.2)* and  *ReLU* activation.
3. **embedding\_component**:
* A Linear layer: *embedding\_size* $\rightarrow$ *embedding\_size // 2*.
* *Dropout(0.2)* and  ReLU.
* A Linear layer: \texttt{embedding\_size // 2} $\rightarrow$ 512.
* *Dropout(0.2)* and  *ReLU*.
4. **Concatenation torch.hstack**:
* The outputs from the *loss\_component(projection\_size)* and the *embedding\_component(512)* are concatenated into a single vector of size *2 \* projection\_size*.
5. **attack\_encoding**:
* A series of 6 fully connected *Linear* layers with *Dropout(0.2)* and  *ReLU* activations between them: *2 \* projection\_size* $\rightarrow$ 512 $\rightarrow$ 256 $\rightarrow$ 128 $\rightarrow$ 64 $\rightarrow$ 32.
* The final linear layer: 32 $\rightarrow$ 2 (output logits for classification).
* A final  *ReLU* activation after the last layer.

7. **Output**
* The model returns the logits (size 2).
* If labels are provided, it also calculates and returns the cross-entropy loss.

After loss and embeddings calculation we should put this together to one dataset and calculate z-score statistics for training FiMMIA neural network.

---

##### Create dataset for training FiMMIA neural network

**Run command**

```bash
fimmia mds-dataset \
  --save_dir="path/to/save/mds/dataset" \
  --model_name="Qwen2.5-VL-3B-Instruct" \
  --origin_df_path="path/to/train.csv" \
  --labels="0,1"
```

Here
* `save_dir` - path for saving merged dataset
* `model_name` - name of MLLM model (using for store results)
* `shuffle` - not shuffle data `0` or shuffle `1`
* `labels` - list of labels in dataset
* `modality_key` - modality column name (`video`, `image`, or `audio`) for loading modality embeds; must match the one used in embedding generation
* `single_file` - run on single file or batches

After running this command we get dataset in mds format with losses and embedding.

In [3]:
from fimmia.utils.data import get_mean_std, get_streaming_ds

In [ ]:
ds = get_streaming_ds("data/train_mds")

In [ ]:
for x in ds:
    break

In [11]:
x

{'answer': 'D',
 'ds_name': 'CommonVideoQA',
 'embedding_input': array([-0.00378995,  0.00234417, -0.00746693, ...,  0.00404607,
        -0.00334608,  0.00647654]),
 'hash': '-1220520432446283628',
 'input': 'Видеофайл: <video>\nВопрос:\nКаким будет наиболее вероятное следующее действие?\n\nA. Очистка огурца от пищевой пленки.\nB. Мытье огурца.\nC. Нарезка огурца.\nD. Нарезка салата.',
 'label': 0,
 'loss_input': 1.46,
 'model_name': 'Qwen2.5-VL-3B-Instruct',
 'neighbor': 'Видеофайл: <video> Вопрос: Каким будет наиболее _эффективным_ следующее действие? A. Очистка _огурца_ от щипцов _и_  пленки. B. Мытье огурца. C. Нарезка огурца. D. Нарезка салата.',
 'num_part': 0,
 'video': 'data/CommonVideoQA/samples/video035.mp4'}

**For test dataset `data/test.csv` apply all steps**:
* [Neighbor generation](#Neighbor-generation)
* [Embedding generation](#Embedding-generation)
* [Loss computation](#Loss-computation)

---

##### calculate z-score statistics

In [ ]:
from fimmia.utils.utils import save_json, load_json

sigmas = get_mean_std(["data/train_mds"])
save_json(sigmas, "data/train_sigmas.json")

In [ ]:
ls /home/jovyan/emelyanov/data_leakage_detect/data/test.csv

In [21]:
sigmas = load_json("data/train_sigmas.json")
sigmas

{'CommonVideoQA_Qwen2.5-VL-3B-Instruct': {'mean': 0.009537732455924316,
  'std': 0.15681258803478634}}

---

##### Training FiMMIA neural network

After data preparation run training of an attack model neural network FiMMIA:

```bash
fimmia train \
  --train_dataset_path="data/train_mds" \
  --model_name="FiMMIABaseLineModelLossNormSTDV2" \
  --output_dir="data/FiMMIA-Video" \
  --num_train_epochs=10 \
  --optim="adafactor" \
  --learning_rate=0.00005 \
  --max_grad_norm=10 \
  --warmup_ratio=0.03 \
  --sigmas_path="data/train_sigmas.json" \
  --sigmas_type="std"
```

Here
* `train_dataset_path` - path to train mds dataset
* `val_dataset_path` - path to test mds dataset
* `model_name` - name FiMMIA neural network architecture
* `num_train_epochs` - number of training epochs
* `output_dir` - path to save FiMMIA model
* `optim` - pytorch optimizer name
* `learning_rate` - learning rate
* `max_grad_norm` - max gradient normalization
* `warmup_ratio` - warmup ratio for optimization
* `sigmas_path` - path for dict with normalization parameters
* `sigmas_type` - type of normalization

As the result we get FiMMIA neural network in directory `data/FiMMIA-Video`. Now we can load model and start Membership Inference Attacks against Large Multimodal LLMs.

In [4]:
from fimmia.fimmia_inference import ModelArguments, init_model

In [7]:
model_args = ModelArguments(
    model_name="FiMMIABaseLineModelLossNormSTDV2",
    model_path="data/FiMMIA-Video"
)
model = init_model(model_args)

In [8]:
model

FiMMIABaseLineModelLossNormSTDV2(
  (loss_component): Sequential(
    (0): Linear(in_features=1, out_features=512, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): ReLU()
  )
  (embedding_component): Sequential(
    (0): Linear(in_features=4096, out_features=2048, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=2048, out_features=512, bias=True)
    (4): Dropout(p=0.2, inplace=False)
    (5): ReLU()
  )
  (attack_encoding): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): Dropout(p=0.2, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=64, bias=True)
    (10): Dropout(p=0.2, inplace=False)
    (11): ReLU()
    (12): Linear(in_features=64, o

### Run MIA inference

Our pretrained models available on 🤗 HuggingFace [FiMMIA collection](https://huggingface.co/collections/ai-forever/fimmia). For Video we download the following model:

```bash
git clone https://huggingface.co/ai-forever/FiMMIA-Video
```

For inference FiMMIA model on new data we should **run command**:

```bash
fimmia infer \
  --model_name="FiMMIABaseLineModelLossNormSTDV2" \
  --model_path="FiMMIA-Video" \
  --test_path="data/train_mds" \
  --save_path="data/predictions.csv" \
  --save_metrics_path="data/metrics.csv" \
  --sigmas_path="data/train_sigmas.json" \
  --sigmas_type="std"
```
Here
* `model_name` - name FiMMIA neural network architecture
* `model_path` - path to load FiMMIA model
* `test_path` - path to test dataset
* `save_path` - path to save predictions
* `save_metrics_path` - path to save metrics

The running command will produce file with prdictions for our dataset for each neighbor and metrics file (if in dataset labels 0, 1).

In [17]:
from fimmia.utils.metrics import get_sample_scores, convert_str

In [20]:
preds = pd.read_csv("data/predictions.csv")
preds["score"] = preds["score"].apply(convert_str)

In [11]:
metrics = pd.read_csv("data/metrics.csv")

In [10]:
preds.head()

,ds_name,hash,label,score
0,CommonVideoQA,-2819276857036119732,0,[0. 0.5009828]
1,CommonVideoQA,-2819276857036119732,0,[0.543804 0.08863425]
2,CommonVideoQA,-2819276857036119732,0,[1.0642736 0. ]
3,CommonVideoQA,-2819276857036119732,0,[0.5575552 0.09166689]
4,CommonVideoQA,-2819276857036119732,0,[1.1353273 0. ]


In [12]:
metrics

,method,auroc,fpr95,tpr05,acc,per_neighbors_acc
0,smia,100.0%,0.0%,100.0%,0.985417,0.8783


For getting predictions for each sample in dataset we should aggregate neighbor scores

$$A(t, m) = \frac{1}{K}\sum_{k=1}^{K}f_{FiMMIA}({\Delta \mathcal{L}^k_{norm}}, \Delta e^k)$$

In [23]:
cscores, labels, y_true, y_pred = get_sample_scores(preds)

  0%|          | 2/4264 [00:00<08:59,  7.90it/s]


In [26]:
print(cscores[:5])

[0.2637179046869278, 0.12685959537823996, 0.3438585201899211, 0.2937004665533702, 0.035671127339204155]


If we wonna make a desicion we can $A(t,m) > 0.5$ -> leak.